# Command finder: gate test

**Idea:** describe what you want in plain English → CLM picks the command from **~30,000 tldr-pages examples** (`common` + `linux`).
This plays to what the smoke tests showed CLM is good at: *"which of these is this about?"* over a large candidate set.

**Test:** 20 queries written in everyday phrasing *before any model ran*, with gold answers defined by regex on the command.
12 plain lookups + 4 **polarity pairs** (keep vs trash, stage vs unstage, local vs remote, upload vs download), which probe the "can't read 'not'" weakness.

**Baselines:** `clm-raw` (raw Qwen3-8B cosine, no trained heads) and **BM25** keyword search. If CLM can't beat keyword search, it isn't a CLM project.

**Pre-declared bar:** `clm-latest` gets **≥ 16/20 in top-5** AND its **top-1 beats both** `clm-raw` and BM25.
Polarity pairs are reported separately. They're the honest "where it breaks" part of the demo, not part of the bar.

Runtime: A100. If the servers from earlier notebooks are still up, setup is skipped.


In [1]:
import os, subprocess, time, json, statistics, re
import requests

SKIP_SETUP = bool(os.environ.get("CLM_SMOKE_SKIP_SETUP"))   # dev only: reuse an already-running server

if not SKIP_SETUP:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True)
    assert smi.returncode == 0, "No GPU. Runtime -> Change runtime type -> L4 GPU"
    gpu_name, total_mib = [s.strip() for s in smi.stdout.strip().split("\n")[0].split(",")]
    total_gb = int(total_mib) / 1024
    # Qwen3-8B bf16 is ~16.4 GB of weights; leave room for KV + clm-serve's heads and vector cache.
    VLLM_UTIL = round(min(0.85, 21.0 / total_gb), 2)
    print(f"{gpu_name}: {total_gb:.1f} GB -> vLLM gpu-memory-utilization {VLLM_UTIL}")
    assert total_gb > 20, "Need a >=24 GB GPU (L4 / A100). A T4 (16 GB) will not fit Qwen3-8B in bf16."


NVIDIA A100-SXM4-40GB: 40.0 GB -> vLLM gpu-memory-utilization 0.53


In [2]:
# ~3-5 min. vLLM + torch go into their OWN Python 3.12 environment, because installing them over
# Colab's preinstalled stack (Python 3.13, its own torch) breaks vLLM on import.
# The notebook kernel only needs the tiny CLM client (plain HTTP, no torch), installed without deps.
VENV = "/content/clmenv"
if not SKIP_SETUP and not os.path.exists(f"{VENV}/bin/vllm"):
    !pip install -q uv
    !uv venv -q {VENV} --python 3.12
    !uv pip install -q --python {VENV}/bin/python contrastive-lm
    !pip install -q --no-deps contrastive-lm
    !{VENV}/bin/python -c "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.6 MB/s eta 0:00:00
vllm 0.30.0 | torch 2.13.0+cu130 | cuda True


In [3]:
def wait_for(url, name, timeout_s, proc=None, log=None):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(f"{name} exited early. Last log lines:\n" + open(log).read()[-3000:])
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print(f"{name} up after {time.time() - t0:.0f}s"); return
        except requests.RequestException:
            pass
        time.sleep(10)
    raise TimeoutError(f"{name} not up after {timeout_s}s; see {log}")

def is_up(url):
    try: return requests.get(url, timeout=2).status_code == 200
    except requests.RequestException: return False

if not SKIP_SETUP and not is_up("http://127.0.0.1:8090/health"):
    # Mirrors the repo's serve_qwen3_8b.sh (eager mode, prefix caching, short max len), with util sized for this GPU.
    vllm_log = open("vllm.log", "w")
    vllm_proc = subprocess.Popen(
        [f"{VENV}/bin/vllm", "serve", "Qwen/Qwen3-8B", "--served-model-name", "qwen3-8b", "--runner", "pooling",
         "--enforce-eager", "--enable-prefix-caching", "--max-model-len", "2048",
         "--gpu-memory-utilization", str(VLLM_UTIL), "--max-num-seqs", "32", "--port", "8090"],
        stdout=vllm_log, stderr=subprocess.STDOUT)
    print("vLLM starting: downloads ~16 GB of weights on first run, usually 5-15 min...")
    wait_for("http://127.0.0.1:8090/health", "vLLM encoder", 30 * 60, vllm_proc, "vllm.log")


vLLM starting: downloads ~16 GB of weights on first run, usually 5-15 min...
vLLM encoder up after 120s


In [4]:
if not SKIP_SETUP and not is_up("http://127.0.0.1:8700/health"):
    clm_log = open("clm.log", "w")
    clm_proc = subprocess.Popen([f"{VENV}/bin/clm-serve", "--port", "8700"], stdout=clm_log, stderr=subprocess.STDOUT)
    wait_for("http://127.0.0.1:8700/health", "clm-serve", 10 * 60, clm_proc, "clm.log")
    print(open("clm.log").read())

from clm import CLMClient, Choice, Score
client = CLMClient("http://127.0.0.1:8700")
print([m["name"] for m in client.models()])
print(requests.get("http://127.0.0.1:8700/health").json())
MODELS = ["clm-raw", "clm-latest"]


clm-serve up after 10s
[clm] models ['clm-latest', 'clm-raw'] on cuda
[clm] embedder http://127.0.0.1:8090/v1/embeddings (qwen3-8b) up; auth off
[clm] vector cache 848.1 MB reserved on cuda (362,354x512d + 6,470x4096d)
[clm] POST http://0.0.0.0:8700/v1/systemone
[clm] playground http://localhost:8700/

['clm-latest', 'clm-raw']
{'ok': True, 'embedder': True, 'models': ['clm-latest', 'clm-raw'], 'cache': {'device': 'cuda', 'reserved_mb': 848.1, 'hit_rate': None, 'pools': {'512': {'dim': 512, 'capacity': 362354, 'used': 0, 'reserved_mb': 742.1, 'hits': 0, 'misses': 0, 'evictions': 0, 'hit_rate': None}, '4096': {'dim': 4096, 'capacity': 6470, 'used': 0, 'reserved_mb': 106.0, 'hits': 0, 'misses': 0, 'evictions': 0, 'hit_rate': None}}}}


## Catalog + queries

In [5]:
!pip install -q rank_bm25
if not os.path.isdir("tldr"):
    !git clone -q --depth 1 https://github.com/tldr-pages/tldr.git
TLDR_COMMIT = subprocess.run(["git", "-C", "tldr", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("tldr-pages @", TLDR_COMMIT, "(CC BY 4.0)")


tldr-pages @ 106eb6eb (CC BY 4.0)


In [6]:
# catalog.py, inlined so the notebook is self-contained
"""tldr-pages -> command catalog, plus the gate's test queries. Shared by the gate notebook and the app."""
import glob
import os
import re

PLATFORMS = ("common", "linux")


def parse_page(path):
    """-> list of {tool, desc, cmd} from one tldr page."""
    tool, out, desc = None, [], None
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        if line.startswith("# "):
            tool = line[2:].strip()
        elif line.startswith("- "):
            desc = line[2:].strip().rstrip(":")
        elif line.startswith("`") and line.endswith("`") and desc:
            out.append({"tool": tool, "desc": desc, "cmd": line[1:-1]})
            desc = None
    return out


def load_catalog(tldr_root):
    seen, cat = set(), []
    for plat in PLATFORMS:
        for p in sorted(glob.glob(os.path.join(tldr_root, "pages", plat, "*.md"))):
            for e in parse_page(p):
                text = f"{e['desc']}: {e['cmd']}"   # what the action head embeds
                if text not in seen:
                    seen.add(text)
                    cat.append({**e, "platform": plat, "text": text})
    return cat


# ---------------------------------------------------------------------------------------------
# Test queries. Written in everyday developer phrasing (NOT tldr's wording), before any model ran.
# gold: regex on the command string; a hit anywhere in top-k counts.
# pair: polarity pairs share an id; the two halves ask for opposite things with near-identical topic.
QUERIES = [
    # plain lookups
    ("disk_free",   "how much disk space is left on my machine",            r"^df\b", None),
    ("port_owner",  "find out which process is using port 8080",            r"^(lsof -i|netstat\b|fuser .*/tcp|ss .*(src|dst|listening|:\{\{|port))", None),
    ("untar",       "unzip a .tar.gz file",                                 r"^tar \S*x", None),
    ("count_lines", "count the number of lines in a file",                  r"^wc .*\[-l\|--lines\]", None),
    ("grep_rec",    "search for a word in every file under this folder",    r"^(grep .*(-r|recursive)|rg\b|ag\b|ack\b)", None),
    ("chmod_x",     "make my script runnable",                              r"^chmod .*\+x", None),
    ("branch_mv",   "rename the current git branch",                        r"^git branch .*(-m|move)", None),
    ("blame",       "see who last edited each line of a file",              r"^git blame(\s|$)", None),
    ("download",    "download a file from a URL",                           r"^(wget\b|curl .*(-O|-o|remote-name|output))", None),
    ("folder_size", "how big is this folder",                               r"^(du|dust|ncdu|gdu)\b", None),
    ("docker_ps",   "list the docker containers that are running",          r"^docker (compose )?(\{\{\[)?(ps|container ls)", None),
    ("venv",        "create a python virtual environment",                  r"(-m venv|^virtualenv\b|^uv venv|^python.*venv)", None),
    # polarity pairs: same topic, opposite intent
    ("undo_keep",   "undo my last commit but keep my changes",              r"^git reset (HEAD~|.*--soft|.*--mixed)", "P1"),
    ("undo_trash",  "undo my last commit and throw the changes away completely", r"^git reset .*--hard", "P1"),
    ("stage_all",   "stage all my changes in git",                          r"^git add .*(-A|all|\{\{\.\}\}|\s\.$)", "P2"),
    ("unstage_all", "unstage everything I added in git",                    r"^git (reset$|restore .*staged)", "P2"),
    ("br_del_loc",  "delete a git branch on my machine",                    r"^git branch .*(-d|-D|delete)", "P3"),
    ("br_del_rem",  "delete a git branch on the remote",                    r"^git push .*(delete|:)", "P3"),
    ("scp_up",      "copy a file from my laptop to a remote server",        r"^scp .*local.* .*remote", "P4"),
    ("scp_down",    "copy a file from a remote server to my laptop",        r"^scp .*remote\S*:\S+ .*local", "P4"),
]


def is_gold(cmd, pattern):
    return re.search(pattern, cmd) is not None


In [7]:
import numpy as np, pandas as pd
from rank_bm25 import BM25Okapi

cat = load_catalog("tldr")
CANDS = [c["text"] for c in cat]
IDX = {t: i for i, t in enumerate(CANDS)}
print(f"{len(cat):,} candidates from {len({c['tool'] for c in cat}):,} tools")

tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in CANDS])

S = requests.Session()
def clm_rank(query, model, k=5):
    t0 = time.perf_counter()
    r = S.post("http://127.0.0.1:8700/v1/rank", json={"context": query, "answers": CANDS, "model": model}, timeout=1800)
    r.raise_for_status()
    rt = (time.perf_counter() - t0) * 1000
    top = [(IDX[x["candidate"]], x["prob"]) for x in r.json()["ranked"][:k]]
    return top, float(r.headers["X-CLM-Latency-Ms"]), rt

def bm25_rank(query, k=5):
    s = bm25.get_scores(tok(query))
    return [(int(i), float(s[i])) for i in np.argsort(-s)[:k]]


29,852 candidates from 6,627 tools


## Warm-up: first call embeds the whole catalog (cold), then it's cached

In [8]:
WARM = {}
for m in ["clm-latest", "clm-raw"]:
    t0 = time.time()
    clm_rank("list files", m)
    WARM[m] = round(time.time() - t0, 1)
    print(f"{m}: cold catalog encode + first query {WARM[m]} s")


clm-latest: cold catalog encode + first query 122.0 s
clm-raw: cold catalog encode + first query 6.9 s


## Evaluation

In [9]:
METHODS = ["bm25", "clm-raw", "clm-latest"]
rows = []
for qid, q, pat, pair in QUERIES:
    row = {"id": qid, "pair": pair or "", "query": q}
    for m in METHODS:
        top = bm25_rank(q) if m == "bm25" else clm_rank(q, m)[0]
        hits = [is_gold(cat[i]["cmd"], pat) for i, _ in top]
        row[m + "@1"] = hits[0]
        row[m + "@5"] = any(hits)
        row[m + "_top1"] = cat[top[0][0]]["cmd"][:60]
        if m == "clm-latest":
            row["p_top1"] = round(top[0][1], 3)
            row["top5"] = [(cat[i]["cmd"][:50], round(p, 3)) for i, p in top]
    rows.append(row)
res = pd.DataFrame(rows)

def score(df):
    return {m: {"top1": int(df[m + "@1"].sum()), "top5": int(df[m + "@5"].sum()), "n": len(df)} for m in METHODS}

SCORES = {"all": score(res), "plain": score(res[res.pair == ""]), "polarity": score(res[res.pair != ""])}
PAIRS_BOTH = {m: int(sum(res[res.pair == p][m + "@1"].all() for p in sorted(set(res.pair) - {""}))) for m in METHODS}

a = SCORES["all"]
GATE = {"top5_ge_16": a["clm-latest"]["top5"] >= 16,
        "top1_beats_raw": a["clm-latest"]["top1"] > a["clm-raw"]["top1"],
        "top1_beats_bm25": a["clm-latest"]["top1"] > a["bm25"]["top1"]}
GATE["PASS"] = all(GATE.values())

pd.set_option("display.max_colwidth", 60)
display(res[["id", "pair"] + [m + s for m in METHODS for s in ("@1", "@5")] + ["clm-latest_top1", "p_top1"]])
display(pd.DataFrame({k: {m: f"{v[m]['top1']}/{v[m]['n']} @1, {v[m]['top5']}/{v[m]['n']} @5" for m in METHODS}
                      for k, v in SCORES.items()}))
print("polarity pairs with BOTH halves right at top-1:", PAIRS_BOTH, "(of 4)")
print("GATE:", GATE)


,id,pair,bm25@1,bm25@5,clm-raw@1,clm-raw@5,clm-latest@1,clm-latest@5,clm-latest_top1,p_top1
0,disk_free,,False,False,False,False,False,False,rmpc,0.113
1,port_owner,,False,True,False,False,False,False,telnet {{host}},0.025
2,untar,,False,False,False,False,False,False,mkisofs -o {{filename.iso}} {{path/to/source_directory}},0.022
3,count_lines,,True,True,False,False,False,False,tldr mv,0.119
4,grep_rec,,False,False,False,False,False,False,nnn,0.030
5,chmod_x,,False,False,False,False,False,False,vagrant init,0.184
6,branch_mv,,False,True,False,False,False,False,pio platform update,0.085
7,blame,,False,False,False,False,False,False,ls,0.089
8,download,,False,False,False,False,False,False,psexec.py -hashes {{LM_Hash}}:{{NT_Hash}} {{domain}}/{{u...,0.021
9,folder_size,,False,False,False,False,False,False,pdftocairo {{path/to/file.pdf}} -jpeg,0.068


,all,plain,polarity
bm25,"8/20 @1, 12/20 @5","3/12 @1, 5/12 @5","5/8 @1, 7/8 @5"
clm-raw,"0/20 @1, 0/20 @5","0/12 @1, 0/12 @5","0/8 @1, 0/8 @5"
clm-latest,"0/20 @1, 1/20 @5","0/12 @1, 1/12 @5","0/8 @1, 0/8 @5"


polarity pairs with BOTH halves right at top-1: {'bm25': 1, 'clm-raw': 0, 'clm-latest': 0} (of 4)
GATE: {'top5_ge_16': False, 'top1_beats_raw': False, 'top1_beats_bm25': False, 'PASS': False}


In [10]:
# The polarity pairs up close: does the top answer flip when the intent flips?
for p in sorted(set(res.pair) - {""}):
    for _, r in res[res.pair == p].iterrows():
        print(f"[{p}] {r['query']}")
        for cmd, prob in r["top5"][:3]:
            print(f"     {prob:6.3f}  {cmd}")
    print()


[P1] undo my last commit but keep my changes
      0.093  pulumi new
      0.070  p4 add
      0.052  sfdk scrape --keep
[P1] undo my last commit and throw the changes away completely
      0.273  git reset --soft HEAD~2
      0.074  git restore {{path/to/file}}
      0.025  git mv {{path/to/file}} {{path/to/destination}}

[P2] stage all my changes in git
      0.035  fj issue create
      0.033  pulumi stack init
      0.027  git mergetool
[P2] unstage everything I added in git
      0.066  git gui
      0.039  yakuake
      0.037  fj issue create

[P3] delete a git branch on my machine
      0.096  tldr mv
      0.032  youtube-viewer --login
      0.022  systemsettings
[P3] delete a git branch on the remote
      0.079  youtube-viewer --login
      0.036  git maintenance unregister
      0.036  git maintenance register

[P4] copy a file from my laptop to a remote server
      0.119  vagrant init
      0.097  ls
      0.015  tldr mv
[P4] copy a file from a remote server to my laptop
 

## Latency with a warm cache (~30k candidates per query)

In [11]:
srv, rt = [], []
for q in ["kill the process on port 3000", "show hidden files", "zip a folder", "check my git history",
          "find big files", "restart nginx", "see memory usage", "follow a log file", "ssh into a server", "pull latest code"]:
    _, s, r = clm_rank(q + f" ({time.time():.0f})", "clm-latest")   # fresh state each time, cached catalog
    srv.append(s); rt.append(r)
LATENCY = {"candidates": len(CANDS), "server_p50_ms": round(float(np.median(srv)), 1),
           "round_trip_p50_ms": round(float(np.median(rt)), 1), "cold_s": WARM}
print(LATENCY)


{'candidates': 29852, 'server_p50_ms': 137.6, 'round_trip_p50_ms': 260.1, 'cold_s': {'clm-latest': 122.0, 'clm-raw': 6.9}}


In [12]:
summary = {"tldr_commit": TLDR_COMMIT, "gate": GATE, "scores": SCORES, "polarity_pairs_both_top1": PAIRS_BOTH,
           "latency": LATENCY,
           "clm_latest_misses_top5": [{"id": r["id"], "query": r["query"], "top1": r["clm-latest_top1"], "p": r["p_top1"]}
                                      for _, r in res.iterrows() if not r["clm-latest@5"]],
           "polarity_top1": {r["id"]: {m: r[m + "_top1"] for m in METHODS} for _, r in res[res.pair != ""].iterrows()}}
res.to_json("gate_results.json", orient="records", indent=1)
print("=" * 20, "PASTE THIS BACK", "=" * 20)
print(json.dumps(summary, indent=1))


==================== PASTE THIS BACK ====================
{
 "tldr_commit": "106eb6eb",
 "gate": {
  "top5_ge_16": false,
  "top1_beats_raw": false,
  "top1_beats_bm25": false,
  "PASS": false
 },
 "scores": {
  "all": {
   "bm25": {
    "top1": 8,
    "top5": 12,
    "n": 20
   },
   "clm-raw": {
    "top1": 0,
    "top5": 0,
    "n": 20
   },
   "clm-latest": {
    "top1": 0,
    "top5": 1,
    "n": 20
   }
  },
  "plain": {
   "bm25": {
    "top1": 3,
    "top5": 5,
    "n": 12
   },
   "clm-raw": {
    "top1": 0,
    "top5": 0,
    "n": 12
   },
   "clm-latest": {
    "top1": 0,
    "top5": 1,
    "n": 12
   }
  },
  "polarity": {
   "bm25": {
    "top1": 5,
    "top5": 7,
    "n": 8
   },
   "clm-raw": {
    "top1": 0,
    "top5": 0,
    "n": 8
   },
   "clm-latest": {
    "top1": 0,
    "top5": 0,
    "n": 8
   }
  }
 },
 "polarity_pairs_both_top1": {
  "bm25": 1,
  "clm-raw": 0,
  "clm-latest": 0
 },
 "latency": {
  "candidates": 29852,
  "server_p50_ms": 137.6,
  "round_trip_p5

In [13]:
# ==== DIAG CELL 1: hubness. Which candidates show up in the top-10 of many *different* queries? ====
from collections import Counter
hub = Counter()
for qid, q, pat, pair in QUERIES:
    top, _, _ = clm_rank(q, "clm-latest", k=10)
    hub.update(cat[i]["text"] for i, _ in top)
print("Candidates in the top-10 of many different queries (hubs):")
for t, n in hub.most_common(8):
    print(f"  {n:2d}/20  {t[:90]}")
HUBS = [(t[:70], n) for t, n in hub.most_common(5)]

Candidates in the top-10 of many different queries (hubs):
   9/20  View documentation for the original command: tldr mv
   7/20  Create a new project using a template: pulumi new
   5/20  [Interactive] Get a list of files on the remote machine: ls
   5/20  Log in to your YouTube account: youtube-viewer --login
   4/20  Open an interactive prompt to check personal mail: mail
   4/20  Create a `Vagrantfile` in the current directory with the base Vagrant box: vagrant init
   4/20  Create a new stack using an isolated deployment target: pulumi stack init
   3/20  Open the settings GUI: systemsettings


In [14]:
# ==== DIAG CELL 2: small candidate sets. Bug or hubness? And does retrieve-then-rerank work? ====
import random
rng = random.Random(0)
K = 50

def rank_subset(query, idxs, model):
    r = S.post("http://127.0.0.1:8700/v1/rank", json={"context": query, "answers": [CANDS[i] for i in idxs], "model": model}, timeout=600)
    r.raise_for_status()
    return [IDX[x["candidate"]] for x in r.json()["ranked"]], float(r.headers["X-CLM-Latency-Ms"])

rows2, lat2 = [], []
for qid, q, pat, pair in QUERIES:
    gold = [i for i, c in enumerate(cat) if is_gold(c["cmd"], pat)]
    gold_set = set(gold)
    bm_scores = bm25.get_scores(tok(q))
    sets = {
        # realistic pipeline: keyword search retrieves 50, CLM re-ranks them
        "bm25_top50": [int(i) for i in np.argsort(-bm_scores)[:K]],
        # isolates CLM's judgment: every correct answer + random distractors (no retrieval step)
        "oracle": gold + rng.sample([i for i in range(len(cat)) if i not in gold_set], K),
    }
    row = {"id": qid, "pair": pair or "", "query": q}
    for sname, idxs in sets.items():
        orders = {"bm25": sorted(idxs, key=lambda i: -bm_scores[i])}
        for m in ["clm-raw", "clm-latest"]:
            orders[m], ms = rank_subset(q, idxs, m)
            lat2.append(ms)
        for meth, order in orders.items():
            row[f"{sname}|{meth}@1"] = order[0] in gold_set
            row[f"{sname}|{meth}@5"] = any(i in gold_set for i in order[:5])
        row[f"{sname}|clm-latest_top1"] = cat[orders["clm-latest"][0]]["cmd"][:55]
    rows2.append(row)
diag = pd.DataFrame(rows2)

DIAG = {}
for sname in ["oracle", "bm25_top50"]:
    DIAG[sname] = {meth: f"{int(diag[f'{sname}|{meth}@1'].sum())}/20 @1, {int(diag[f'{sname}|{meth}@5'].sum())}/20 @5"
                   for meth in ["bm25", "clm-raw", "clm-latest"]}
DIAG["bm25_recall@50"] = f"{int(diag['bm25_top50|bm25@5'].sum())}/20 in top-5, " + \
                         f"{sum(any(is_gold(cat[i]['cmd'], pat) for i in [int(j) for j in np.argsort(-bm25.get_scores(tok(q)))[:K]]) for _, q, pat, _ in QUERIES)}/20 in top-50"
DIAG["rerank_server_p50_ms"] = round(float(np.median(lat2)), 1)
h1 = int(diag["bm25_top50|clm-latest@1"].sum())
DIAG["hybrid_gate"] = {"clm_rerank_top1_ge_11": h1 >= 11,
                       "beats_raw_rerank": h1 > int(diag["bm25_top50|clm-raw@1"].sum())}

display(pd.DataFrame({k: v for k, v in DIAG.items() if k in ("oracle", "bm25_top50")}))
print("\nPolarity pairs, BM25 top-50 -> CLM re-rank, top-1:")
for _, r in diag[diag.pair != ""].iterrows():
    print(f"  [{r['pair']}] {'OK ' if r['bm25_top50|clm-latest@1'] else 'MISS'} {r['query'][:55]:55} -> {r['bm25_top50|clm-latest_top1']}")

print("=" * 20, "PASTE THIS BACK", "=" * 20)
print(json.dumps({"hubs": HUBS, "diag": DIAG,
                  "rerank_misses": [{"q": r["query"], "top1": r["bm25_top50|clm-latest_top1"]}
                                    for _, r in diag.iterrows() if not r["bm25_top50|clm-latest@1"]]}, indent=1))

,oracle,bm25_top50
bm25,"18/20 @1, 20/20 @5","8/20 @1, 12/20 @5"
clm-raw,"5/20 @1, 11/20 @5","0/20 @1, 2/20 @5"
clm-latest,"13/20 @1, 17/20 @5","2/20 @1, 6/20 @5"



Polarity pairs, BM25 top-50 -> CLM re-rank, top-1:
  [P1] OK  undo my last commit but keep my changes                 -> git reset --soft HEAD~2
  [P1] MISS undo my last commit and throw the changes away complete -> git reset --soft HEAD~2
  [P2] MISS stage all my changes in git                             -> git diff
  [P2] MISS unstage everything I added in git                       -> git status
  [P3] MISS delete a git branch on my machine                       -> git sync
  [P3] MISS delete a git branch on the remote                       -> git sync
  [P4] MISS copy a file from my laptop to a remote server           -> sshfs {{username}}@{{remote_host}}:{{remote_directory}}
  [P4] MISS copy a file from a remote server to my laptop           -> sshfs {{username}}@{{remote_host}}:{{remote_directory}}
==================== PASTE THIS BACK ====================
{
 "hubs": [
  [
   "View documentation for the original command: tldr mv",
   9
  ],
  [
   "Create a new project using a te

In [15]:
from clm import Noul, Choice, Score
r = client.system_one(
    state="Customer: my invoice was charged twice and nobody answers the phone!",
    questions={"urgency": Noul(instructions="Is this urgent?"),
               "department": Choice(instructions="Which team should handle this?",
                                    criteria={"billing": "Charges, invoices, refunds", "technical": "Bugs and outages"}),
               "frustration": Score(instructions="How frustrated is the customer?",
                                    criteria=["Calm", "Frustrated", "Very angry"])})
print("ours:  ", round(r.answers["urgency"].noul, 5), r.answers["department"].probabilities, round(r.answers["frustration"].score, 5))
print("README: 0.41022 {'billing': 0.93878, 'technical': 0.06122} 1.98386")
print(client.rank("What causes tides on Earth?", None,
                  ["The Moon's gravitational pull.", "Photosynthesis in plants.", "Because the Earth is round."])[0],
      "| README: prob ~0.997 (model card: 0.993)")

ours:   0.83947 {'billing': 0.9891176412298573, 'technical': 0.010882358770142674} 1.99998
README: 0.41022 {'billing': 0.93878, 'technical': 0.06122} 1.98386
{'rank': 1, 'candidate': "The Moon's gravitational pull.", 'prob': 0.9932577156553689} | README: prob ~0.997 (model card: 0.993)
